# Objective-aware multi-horizon scenario router

This notebook adds a **routing layer above the scenario models**. The Scenario Engine remains an
autonomous predictive-distribution project; the router simply allows different candidate laws to be
selected for different statistical objectives and horizons.

Built-in objectives:

- `joint_distribution` — Energy Score;
- `dependence` — Variogram Score;
- `marginals` — average marginal CRPS;
- `projections` — CRPS along user-defined linear projections;
- `tail_risk` — joint FZ0 VaR/ES score on a scalar loss projection;
- `density` — NLL only when every compared candidate exposes a genuinely comparable predictive density.

The router supports `hard`, paired `one_se`, and `soft` scenario-mixture policies. Selection is
point-in-time: only validation observations strictly prior to the forecast date are admissible.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
SRC=ROOT/'src'
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))

from multi_asset_scenario_engine import ObjectiveRouter, objective_score, mix_scenarios

pd.set_option('display.float_format',lambda x:f'{x:,.6f}')


## 1. Frozen descriptive map — not a nested-router result

The current committed V30 summary is useful for seeing why routing is economically sensible, but it is
**not** itself a point-in-time routing backtest because it contains horizon-level mean scores. The table
below therefore remains descriptive only. A true routing experiment uses per-as-of losses generated by
`03_duration_pit_factorial.ipynb` or by the full repricing validation layer.


In [ ]:
desc=pd.read_csv(ROOT/'results'/'v31_descriptive_objective_map.csv')
display(desc.sort_values(['objective','horizon']))


## 2. Point-in-time score history

After `03_duration_pit_factorial.ipynb` is rerun, it writes `v31_compact_score_history.csv` containing
per-as-of Energy and Variogram losses. We reshape those losses to the router schema below. This compact
gate covers `joint_distribution` and `dependence`; tail-risk routing requires instrument/portfolio loss
scenarios from the full engine rather than the compact driver panel.


In [ ]:
hist_path=ROOT/'results'/'v31_compact_score_history.csv'
if hist_path.exists():
    raw=pd.read_csv(hist_path,parse_dates=['asof'])
    long=[]
    for objective,col in [('joint_distribution','energy_score'),('dependence','variogram_score')]:
        q=raw[['asof','horizon','model',col]].rename(columns={col:'loss'}).copy()
        q['objective']=objective
        long.append(q)
    score_history=pd.concat(long,ignore_index=True)
    display(score_history.head())
else:
    score_history=None
    print('Per-as-of history not committed yet. Run 03_duration_pit_factorial.ipynb first.')


## 3. Router policies

The recommended default is `one_se`: identify the empirical best model, retain candidates whose paired
loss difference is within one standard error, and then choose the simplest eligible specification.
This avoids turning tiny sampling differences into unstable model switches.

`hard` always chooses the lowest historical mean loss. `soft` converts validation performance into model
weights and creates a finite mixture of scenario laws; the mixture must then be **re-scored from its own
scenarios**, because a weighted average of component losses is generally not the proper score of the mixture.


In [ ]:
COMPLEXITY={
    'raw_duration|blockwise|direct_H':0,
    'pit_pooled|blockwise|direct_H':1,
    'raw_duration|hierarchical|direct_H':1,
    'pit_pooled|hierarchical|direct_H':2,
    # recursive candidates can remain in research history but are deliberately more complex
    'raw_duration|blockwise|recursive_1W':2,
    'pit_pooled|blockwise|recursive_1W':3,
    'raw_duration|hierarchical|recursive_1W':3,
    'pit_pooled|hierarchical|recursive_1W':4,
}

if score_history is not None:
    router=ObjectiveRouter(score_history,complexity=COMPLEXITY,min_history=20,lookback=80)
    last=score_history['asof'].max()+pd.Timedelta(days=1)
    rows=[]
    for objective in ['joint_distribution','dependence']:
        for horizon in ['1W','1M','3M','1Y']:
            try:
                d=router.select(last,horizon,objective,policy='one_se')
                rows.append({'objective':objective,'horizon':horizon,'selected':d.selected_model,
                             'empirical_best':d.best_mean_model,'n_history':d.n_history,
                             'gap_best_second':d.gap_best_second,'z_gap':d.z_gap})
            except ValueError as e:
                rows.append({'objective':objective,'horizon':horizon,'selected':None,
                             'empirical_best':None,'n_history':0,'gap_best_second':np.nan,'z_gap':np.nan})
    display(pd.DataFrame(rows))


## 4. Nested walk-forward adjudication

A legitimate router is evaluated as another forecasting model. At each outer date it uses only previous
validation losses, freezes the model choice, and is then scored on the next realised horizon. This cell
runs the nested selector for objectives for which per-date history is available.


In [ ]:
if score_history is not None:
    wf=[]
    for objective in ['joint_distribution','dependence']:
        for horizon in ['1W','1M','3M','1Y']:
            try:
                r=router.walk_forward_select(objective,horizon,policy='one_se')
                if len(r): wf.append(r)
            except ValueError:
                pass
    router_oos=pd.concat(wf,ignore_index=True) if wf else pd.DataFrame()
    display(router_oos.groupby(['objective','horizon']).agg(
        n=('router_loss','size'),mean_router_loss=('router_loss','mean')) if len(router_oos) else router_oos)


## 5. Scoring any scenario law

The scoring API operates directly on scenario draws. This keeps the Scenario Engine independent of any
particular downstream project. A downstream system can also append a custom realised decision loss to
the same history schema and use the router without adding downstream logic to this repository.


In [ ]:
# Example API (illustrative arrays only):
rng=np.random.default_rng(7)
X=rng.normal(size=(1000,4))
y=np.array([0.2,-0.1,0.05,0.15])
labels=np.array(['equity','rates','credit','fx'],dtype=object)

example_scores={
    'joint_distribution':objective_score(X,y,'joint_distribution'),
    'dependence':objective_score(X,y,'dependence',labels=labels),
    'marginals':objective_score(X,y,'marginals'),
    'projections':objective_score(X,y,'projections',projections=np.eye(4)),
    'tail_risk':objective_score(X,y,'tail_risk',loss_projection=np.array([-1.0,0.0,0.0,0.0]),alpha=0.95),
}
pd.Series(example_scores,name='loss')


## Decision rule for promotion

The router is promoted only if the **nested OOS router** improves the relevant objective relative to a
strong fixed-model benchmark without relying on hindsight. If model rankings are unstable or differences
are negligible, the one-SE rule should collapse back toward the simpler candidate rather than manufacture
complexity.
